Diante Calhoun CIS 730 Term Project
This program is designed for the Term Project for CIS 730.
Specfically: Learning Risk-Aware Policies: A Comparative Study of Reinforcement Learning and Heuristic Agents in Flip 7

First using ChatGPT scaffolding - lets create a clean starter structure to build from.

In [ ]:
import random
from dataclasses import dataclass, field
from typing import List, Optional, Union

Card = Union[int, str]


# ----------------------------
# Deck creation
# ----------------------------
def create_flip7_deck() -> List[Card]:
    """
    Simple Flip 7 deck representation.

    Number cards:
      one 1, two 2s, three 3s, ..., twelve 12s

    Special cards:
      Adjust counts here to match your exact rule set later.
    """
    deck: List[Card] = []

    # Number cards
    for value in range(1, 13):
        deck.extend([value] * value)

    # Special cards (placeholder counts — tune later to real rules)
    deck.extend(["freeze"] * 3)
    deck.extend(["draw3"] * 3)
    deck.extend(["second_chance"] * 2)
    deck.extend(["+2"] * 2)
    deck.extend(["+4"] * 2)
    deck.extend(["+6"] * 1)
    deck.extend(["+8"] * 1)
    deck.extend(["+10"] * 1)
    deck.extend(["x2"] * 1)
    deck.extend([0] * 1)  # there is only 1 zero card

    random.shuffle(deck)
    return deck


# ----------------------------
# Player state
# ----------------------------
@dataclass
class PlayerState:
    name: str
    round_points: int = 0
    total_points: int = 0
    drawn_numbers: set = field(default_factory=set)
    cards_drawn_count: int = 0
    busted: bool = False
    stopped: bool = False
    frozen: bool = False
    second_chance_available: bool = False
    multiplier: int = 1

    def is_active(self) -> bool:
        return not self.busted and not self.stopped and not self.frozen

    def reset_round(self) -> None:
        self.round_points = 0
        self.drawn_numbers = set()
        self.cards_drawn_count = 0
        self.busted = False
        self.stopped = False
        self.frozen = False
        self.second_chance_available = False
        self.multiplier = 1


# ----------------------------
# Core game engine
# ----------------------------
class Flip7RoundEngine:
    def __init__(self, player_names: List[str], seed: Optional[int] = None):
        if seed is not None:
            random.seed(seed)

        self.players = [PlayerState(name=n) for n in player_names]
        self.deck: List[Card] = []
        self.turn_index = 0

    def reset_round(self) -> None:
        self.deck = create_flip7_deck()
        self.turn_index = 0
        for player in self.players:
            player.reset_round()

    def draw_card(self) -> Card:
        if not self.deck:
            raise RuntimeError("Deck is empty.")
        return self.deck.pop()

    def apply_card_by_index(self, player_idx: int, card: Card) -> None:
        player = self.players[player_idx]
        player.cards_drawn_count += 1

        if isinstance(card, int):
            if card in player.drawn_numbers:
                if player.second_chance_available:
                    player.second_chance_available = False
                    log(f"{player.name} used Second Chance to avoid busting.")
                else:
                    player.busted = True
                    player.round_points = 0
            else:
                player.drawn_numbers.add(card)
                player.round_points += card

        elif card == "second_chance":
            player.second_chance_available = True
            log(f"{player.name} gained Second Chance.")

        elif card == "freeze":
            target_idx = self.freeze_target_heuristic(player_idx)
            if target_idx is not None:
                self.players[target_idx].frozen = True
                log(f"{player.name} froze {self.players[target_idx].name}.")

        elif card == "draw3":
            target_idx = self.draw3_target_heuristic(player_idx)
            log(f"{player.name} applies Draw 3 to {self.players[target_idx].name}.")
            self.force_draw_cards(target_idx, num_cards=3)

        elif card == "x2":
            player.multiplier *= 2
            log(f"{player.name} multiplier increased to x{player.multiplier}.")

        elif isinstance(card, str) and card.startswith("+"):
            bonus = int(card.replace("+", ""))
            player.round_points += bonus
            log(f"{player.name} gains bonus {bonus} points.")

        else:
            raise ValueError(f"Unknown card: {card}")

    def bank_points(self, player: PlayerState) -> None:
        if not player.busted:
            player.total_points += player.round_points * player.multiplier
        player.stopped = True

    def all_players_done(self) -> bool:
        return all(p.busted or p.stopped or p.frozen for p in self.players)

    def get_leading_opponent(self, current_player_idx: int) -> Optional[int]:
        opponents = [
            (idx, p.total_points + p.round_points)
            for idx, p in enumerate(self.players)
            if idx != current_player_idx
        ]
        if not opponents:
            return None
        return max(opponents, key=lambda x: x[1])[0]

    def print_round_state(self) -> None:
        for p in self.players:
            log(
                f"{p.name}: round={p.round_points}, total={p.total_points}, "
                f"drawn={sorted(p.drawn_numbers)}, busted={p.busted}, "
                f"stopped={p.stopped}, frozen={p.frozen}, "
                f"2nd={p.second_chance_available}, x{p.multiplier}"
            )
    # special card resolutions
    def freeze_target_heuristic(self, current_player_idx: int) -> Optional[int]:
        """
        Freeze the leading opponent.
        """
        return self.get_leading_opponent(current_player_idx)

    def draw3_target_heuristic(self, current_player_idx: int, trailing_threshold: int = 15,
                               opponent_card_threshold: int = 3) -> int:
        """
        If the current player is trailing significantly, apply Draw 3 to self.
        Otherwise, apply Draw 3 to the leading opponent once opponents have
        accumulated enough cards to make pressure meaningful.
        """
        current_player = self.players[current_player_idx]
        current_score = current_player.total_points + current_player.round_points

        leading_idx = self.get_leading_opponent(current_player_idx)

        if leading_idx is None:
            return current_player_idx

        leading_player = self.players[leading_idx]
        leading_score = leading_player.total_points + leading_player.round_points
        score_gap = leading_score - current_score

        if score_gap >= trailing_threshold:
            return current_player_idx

        if leading_player.cards_drawn_count >= opponent_card_threshold:
            return leading_idx

        return current_player_idx
    
    # Adding helper to force card draws for testing - NOT FOR RL POLICY
    def force_draw_cards(self, target_idx: int, num_cards: int = 3) -> None:
        """
        Force a player to draw multiple cards immediately.
        Stops early if the player busts.
        """
        target = self.players[target_idx]

        for _ in range(num_cards):
            if target.busted or target.stopped:
                break

            if not self.deck:
                break

            card = self.draw_card()
            log(f"{target.name} forced draw: {card}")
            self.apply_card_by_index(target_idx, card)

            if target.busted:
                log(f"{target.name} busted during forced draws.")
                break

def total_visible_score(player: PlayerState) -> int:
    return player.total_points + player.round_points

In [ ]:
# Adding logging for initialization and testing
import logging
# force logging to reset
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

def log(msg: str) -> None:
    """Prints to console and writes to log file simultaneously."""
    print(msg)
    logging.info(msg)
        
# Set up training log
logging.basicConfig(
    filename='flip7_training_log4-9-26.txt',
    filemode='w',          # 'w' overwrites each run, change to 'a' to append
    format='%(message)s',
    level=logging.INFO
)
log("="*40)
log("Flip 7 Engine Initialization")
log("="*40)



Quick Test of gameplay

In [ ]:
engine = Flip7RoundEngine(["RL_Agent", "Conservative", "Greedy"])
engine.reset_round()

player = engine.players[0]

for _ in range(5):
    card = engine.draw_card()
    print("Drew:", card)
    engine.apply_card_by_index
    engine.print_round_state()
    if player.busted:
        print("Player busted.")
        break

This broke need to revisit

Addign actual policy for our other players to give our agent better training.

In [ ]:
def conservative_policy(player: PlayerState, stop_threshold: int = 18, max_cards: int = 4) -> str:
    """
    Simple baseline:
    - stop if round score reaches threshold
    - or if enough cards have already been drawn
    - otherwise draw
    """
    if player.round_points >= stop_threshold or player.cards_drawn_count >= max_cards:
        return "STOP"
    return "DRAW"


def greedy_policy(player: PlayerState, chase_cards: int = 7, stop_threshold: int = 30) -> str:
    """
    Greedy baseline:
    - keep drawing while chasing 7 unique cards
    - otherwise stop only at a high score
    """
    if len(player.drawn_numbers) >= chase_cards:
        return "STOP"
    if player.round_points >= stop_threshold:
        return "STOP"
    return "DRAW"


def run_single_turn(engine: Flip7RoundEngine, player_idx: int, action: str) -> None:
    player = engine.players[player_idx]

    if player.busted or player.stopped:
        print(f"{player.name} cannot act (busted={player.busted}, stopped={player.stopped}).")
        return

    print(f"\n{player.name} action: {action}")

    if action == "STOP":
        engine.bank_points(player)
        print(f"{player.name} banks {player.round_points * player.multiplier} points.")
        return

    if action == "DRAW":
        card = engine.draw_card()
        print(f"{player.name} drew: {card}")
        engine.apply_card_by_index(player_idx, card)

        if player.busted:
            print(f"{player.name} busted and loses round points.")
        return

    raise ValueError(f"Unknown action: {action}")

Test-run the engine

In [ ]:
engine = Flip7RoundEngine(["RL_Agent", "Conservative", "Greedy"])
engine.reset_round()

rl_idx = 0
rl_player = engine.players[rl_idx]

# max cards at 4, so it stops after 4 cards. Not the project policy, just a test.
while not rl_player.busted and not rl_player.stopped:
    action = conservative_policy(rl_player, stop_threshold=20, max_cards=4) 
    run_single_turn(engine, rl_idx, action)
    engine.print_round_state()

In [ ]:
# Special card testing
engine = Flip7RoundEngine(["RL_Agent", "Conservative", "Greedy"])
engine.reset_round()

# Manually create some scores/card counts for testing
engine.players[0].round_points = 5 # makes the RL the trailing player - low points
engine.players[1].round_points = 18 # tests the freeze targeting
engine.players[1].cards_drawn_count = 4 # tests the draw 3 targeting
engine.players[2].round_points = 10 # just some points for the third player

print("Before special card:")
engine.print_round_state()

print("\nTesting freeze...")
engine.apply_card_by_index(0, "freeze")
engine.print_round_state()

print("\nTesting draw3...")
engine.apply_card_by_index(0, "draw3")
engine.print_round_state()

In [ ]:
# Add Balanced Policy
# 4.9.26 - increasing stop thresh from 23 to 26 and expanding max cards to 6, to give balanced a better chance.
def balanced_policy(player: PlayerState, stop_threshold: int = 26, max_cards: int = 6) -> str:
    """
    Middle-ground baseline:
    - More risk tolerant than conservative, less aggressive than greedy
    - Stops at a moderate score or card count
    """
    if player.round_points >= stop_threshold or player.cards_drawn_count >= max_cards:
        return "STOP"
    return "DRAW"

In [ ]:
# adding Score bin for our state representation. 4.9.26
def get_score_bin(round_score: int) -> int:
    """
    Converts round score into discrete bins for state representation.
    This reduces state space and focuses on strategic thresholds.
    Small note - Might need to tune these further, but I'm not fully accounting for the *2 multipliers and bonuses.
    """
    # Make 8 really specialized bins, as the mid matters more than early amounts and amounts more than 50.
    if round_score <= 12: return 0
    elif round_score <= 18: return 1  # this is around the conservative stop threshold
    elif round_score <= 23: return 2  # this is around the balanced stop threshold
    elif round_score <= 29: return 3  # this is around the greedy stop threshold
    elif round_score <= 36: return 4  # Risky territory
    elif round_score <= 42: return 5  # very risky, close to busting
    elif round_score <= 49: return 6  # Extremely risky
    else: return 7  # 50+ is all one bin, expecting the agent to learn to stop more before this point.
    

Create Q-Table for game state
* This includes creating Bins that detail the gap of the score for RL vs Opponent

In [ ]:
# scaffolding logic from Claude, actual variables tuned by me after testing and looking at game dynamics. The state includes:
def get_state(player: PlayerState, engine: "Flip7RoundEngine", player_idx: int) -> tuple:
    """
    Converts current game situation into a hashable state tuple for the Q-table.
    Bins the score gap so the state space stays manageable.
    Replaced the raw round score with score_bin.
    """
    # round_score = player.round_points -deprecated in favor of score_bin 4.9.26
    # 4.9.26 - Replacing Round Score with score_bin to reduce state space and focus on strategic thresholds rather than exact points, which can vary widely.
    
    score_bin = get_score_bin(player.round_points)  
    cards_drawn = player.cards_drawn_count
    has_second_chance = int(player.second_chance_available)

    # Score gap vs best opponent (binned into ranges)
    leading_idx = engine.get_leading_opponent(player_idx)
    if leading_idx is not None:
        leading_score = engine.players[leading_idx].total_points + engine.players[leading_idx].round_points
        my_score = player.total_points + player.round_points
        raw_gap = leading_score - my_score  # positive = we're behind
    else:
        raw_gap = 0

    # Bin the gap: behind by a lot, a little, roughly even, or we're ahead
    # 4.10.26 - adjusting bins a bit tighter
    if raw_gap > 45:
        gap_bin = 3    # significantly behind
    elif raw_gap > 15:
        gap_bin = 2    # moderately behind
    elif raw_gap >= -15:
        gap_bin = 1    # roughly even
    else:
        gap_bin = 0    # we're ahead

    return (score_bin, cards_drawn, gap_bin, has_second_chance)

In [ ]:
# Updated Round Loop
def run_round(engine: Flip7RoundEngine, policies: dict) -> None:
    """
    Runs one full round. Players cycle in sequence until all are done.
    policies: dict mapping player index -> callable(player, engine, idx) -> "DRAW" or "STOP"
    """
    engine.reset_round()

    # adding printout of round start and player states for clarity during testing -4/1/26
    # changing to log instead of print for consistency with other outputs - 4/2/26
    log(f"\n{'='*40}")
    log(f"  ROUND START")
    log(f"{'='*40}")

    while not engine.all_players_done():
        for idx, player in enumerate(engine.players):
            if not player.is_active():
                continue

            action = policies[idx](player, engine, idx)

            if action == "STOP":
                engine.bank_points(player)
                log(f"{engine.players[idx].name} STOPPED and banks {player.round_points * player.multiplier} points.")
            elif action == "DRAW":
                card = engine.draw_card()
                log(f"{engine.players[idx].name} drew: {card}") # was missing drawn card in original scaffolding, added for clarity
                engine.apply_card_by_index(idx, card)
                if player.busted:
                    player.stopped = True  # mark done so loop skips them
                    log(f"{engine.players[idx].name} BUSTED and loses round points.")

    # Bank points for frozen players (they keep what they had)
    # Updated logic to freeze per round, not entire game, so we bank points for frozen players at round end.
    for player in engine.players:
        if player.frozen and not player.busted:
            player.total_points += player.round_points * player.multiplier

Updated Game Loop

In [ ]:
def run_game(player_names: list, policies: dict, win_threshold: int = 200,
             max_rounds: int = 50, verbose: bool = False) -> dict:
    """
    Runs a full game until one player reaches win_threshold points.
    Returns a results dict with winner, final scores, and round count.
    """
    engine = Flip7RoundEngine(player_names)
    round_num = 0

    while round_num < max_rounds:
        round_num += 1
        run_round(engine, policies)

        if verbose:
            print(f"\n{'='*40}")
            print(f"  END OF ROUND {round_num} SUMMARY")
            print(f"{'='*40}")
            engine.print_round_state()
            print(f"{'='*40}\n")

        # Check win condition
        for player in engine.players:
            if player.total_points >= win_threshold:
                return {
                    "winner": player.name,
                    "scores": {p.name: p.total_points for p in engine.players},
                    "rounds": round_num
                }

    # If max rounds hit with no winner, highest score wins
    winner = max(engine.players, key=lambda p: p.total_points)
    return {
        "winner": winner.name,
        "scores": {p.name: p.total_points for p in engine.players},
        "rounds": round_num
    }

Quick test to ensure our policies work, and we can run a game

In [ ]:
# Wrap heuristics in the signature run_round expects: (player, engine, idx) -> action
def conservative_agent(player, engine, idx): return conservative_policy(player)
def greedy_agent(player, engine, idx):      return greedy_policy(player)
def balanced_agent(player, engine, idx):    return balanced_policy(player)

policies = {
    0: conservative_agent,
    1: greedy_agent,
    2: balanced_agent,
}

result = run_game(
    player_names=["Conservative", "Greedy", "Balanced"],
    policies=policies,
    win_threshold=200,
    verbose=True
)

log(f"\nWinner: {result['winner']}")
log(f"Final scores: {result['scores']}")
log(f"Rounds played: {result['rounds']}")

# Section 3 - State Representation and Reward Function
Our Game Simulator has been successfully built, so now it's time to start diving into our reward function

My design includes a high learning rate, whic h the agent will update aggressively from each new experience during the game. I expect to see very erratic behavior early as it stabilizes as training progressess.

In terms of rewards, my design is hybrid where I want to reward slightly for banking the points, but a bigger win condition for winning. Very nuanced reward system rewarding  and punishing based on understanding of game state. Very careful in this initial setup, as this game can vary greatly, which could lead to the agent learning the wrong lessons. Let's try it out.

In [ ]:
# Compute Rewards
def compute_reward(points_banked: int, busted: bool, score_gap: int,
                   win_threshold: int = 200) -> float:
    """
    score_gap: leading opponent total - agent total
               positive = agent is BEHIND
               negative = agent is AHEAD
    """
    if busted:
        if score_gap <= -45:      # ahead by 45+ and still drew — greed tax
            return -0.6
        elif score_gap <= -20:    # ahead by 20-44 — moderate penalty
            return -0.45
        else:                     # trailing or close — standard penalty
            return -0.3           # this covers both even AND trailing scenarios

    score_component    = points_banked / 50 # previously 200, now 50 to give more weight to round points since they are more immediate and visible to the agent 4/3/26
    position_component = score_gap / win_threshold
    reward = 0.6 * score_component + 0.4 * (-position_component)

    return min(reward, 1.0) # cap max reward to 1.0 to prevent runaway values from very high round points or large score gaps

# Section 4: Q-Learning Agent

Quick explanation. A Q-table is essentially a dictionary that stores the state tuple for Draw and Stop.
This section contains our hyperparameters. Of concern,  we have an epsilon decay of .995, which means around game 600, our agent should start to make decisions from scenarios learned rather than random ones.
However, because Flip 7 reshuffles every round, we're not hitting a full state coverage in this stochasic environment in 600 games. We likely need to hit something closer to 10,000.

In [ ]:
from collections import defaultdict

## Section 4 — Q-Learning Agent

# Hyperparameters
ALPHA   = 0.7     # learning rate — aggressive updates from new experience
GAMMA   = 0.63    # discount factor — slightly future-leaning
EPSILON = 1.0     # exploration rate — starts fully exploratory
# adjusting the epsilon_decay to be slower than .005, because we really need to hit around 8000 games of training.
EPSILON_DECAY = 0.9996   # multiplied after each game

EPSILON_MIN   = 0.05    # floor — agent never stops exploring entirely

# Q-table
# Structure: {state_tuple: {"DRAW": float, "STOP": float}}
q_table = defaultdict(lambda: {"DRAW": 0.0, "STOP": 0.0})

In [ ]:
import random
# logic from Claude.
def select_action(state: tuple, epsilon: float) -> str:
    """
    Epsilon-greedy action selection.
    With probability epsilon: explore (random action)
    With probability 1-epsilon: exploit (best known action)
    """
    if random.random() < epsilon:
        return random.choice(["DRAW", "STOP"])
    
    # Exploit — pick action with highest Q-value
    q_values = q_table[state]
    if q_values["DRAW"] >= q_values["STOP"]:
        return "DRAW"
    return "STOP"

In [ ]:
# Scaffolding logic from Claude, but reward and state components are my own based on game dynamics and testing.
def update_q_table(state: tuple, action: str, reward: float, 
                   next_state: tuple, terminal: bool) -> None:
    """
    Applies the Bellman update to the Q-table.
    terminal: True when agent busted (no future state to evaluate) 
    """
    current_q = q_table[state][action]
    
    if terminal:
        # No future rewards — update purely from experience
        target = reward
    else:
        # Future rewards factor in via best next action
        best_next_q = max(q_table[next_state].values())
        target = reward + GAMMA * best_next_q
    
    # Nudge current estimate toward target
    q_table[state][action] = current_q + ALPHA * (target - current_q)

## Section 5 - Training

This is pretty much our run_game loop from earlier, with the RL agent plugged in.
Small adjustment made here - Our RL agent needs to know the TOTAL score gap for the reward function to work at the game level as opposed to the round level.

In [ ]:
# jupyter notebook bombed when I tired to run the training loop, so I'm making the print statement to a log file.
import logging
# force logging to reset
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
    
# Set up training log
logging.basicConfig(
    filename='flip7_training_log.txt',
    filemode='w',          # 'w' overwrites each run, change to 'a' to append
    format='%(message)s',
    level=logging.INFO
)
log("="*40)
log("TRAINING RUN START")
log("="*40)

def log(msg: str) -> None:
    """Prints to console and writes to log file simultaneously."""
    print(msg)
    logging.info(msg)

In [ ]:
# Orignial scaffoling for training loop from Claude, but the actual logic and tracking variables are my own based on testing and game dynamics.
# Specifically - The decision to run the game vs 2 or 3 oppenents, number of games to run, and which players to include in the game.
# IF we come up with addition players we can easily add them to the training loop and policies, but for now we want to focus on the 3 player game to give the RL agent a more complex environment to learn in.
def train_rl_agent(num_games: int = 10000, 
                   num_opponents: int = 3,
                   win_threshold: int = 200,
                   verbose_every: int = 1000) -> dict:
    """
    Trains the RL agent via Q-learning against heuristic opponents.
    Returns training statistics for analysis.
    """
    global EPSILON

    # Training tracking
    stats = {
        "wins": 0,
        "losses": 0,
        "busts": 0,
        "win_rate_log": [],      # win rate snapshot every verbose_every games
        "epsilon_log": [],        # epsilon at each snapshot
        "avg_reward_log": []      # average reward at each snapshot
    }

    recent_rewards = []   # rolling window for avg reward tracking

    for game_num in range(1, num_games + 1):

        # --- Game setup ---

        OPPONENT_CONFIGS = {
        2: ["RL_Agent", "Conservative", "Greedy"],
        3: ["RL_Agent", "Conservative", "Greedy", "Balanced"]
        }

        player_names = OPPONENT_CONFIGS.get(num_opponents, OPPONENT_CONFIGS[3])
        
        engine = Flip7RoundEngine(player_names) # silent mode to suppress print during training - 4/2/26
        rl_idx = 0
        game_over = False
        game_reward = 0.0

        while not game_over:

            # --- Round setup ---
            engine.reset_round()
            round_over = False

            while not round_over:

                # Only process RL agent's turn here
                # Opponents are handled separately below
                all_done = True

                for idx, player in enumerate(engine.players):
                    if not player.is_active():
                        continue

                    all_done = False

                    if idx == rl_idx:
                        # --- RL Agent turn ---
                        state = get_state(player, engine, rl_idx)
                        action = select_action(state, EPSILON)
                        #log(f"DEBUG ACTION — RL_Agent selected: {action}, round_points={player.round_points}, stopped={player.stopped}")

                        if action == "STOP":
                            # Compute reward before banking
                            leading_idx = engine.get_leading_opponent(rl_idx)
                            if leading_idx is not None:
                                score_gap = engine.players[leading_idx].total_points - player.total_points
                            else:
                                score_gap = 0

                            engine.bank_points(player)
                            reward = compute_reward(
                                points_banked=player.round_points,
                                busted=False,
                                score_gap=score_gap
                            )
                            next_state = get_state(player, engine, rl_idx)
                            update_q_table(state, action, reward, next_state, terminal=False)

                        elif action == "DRAW":
                            card = engine.draw_card()
                            #log(f"DEBUG — RL_Agent drew: {card}, active before apply: {player.is_active()}")
                            engine.apply_card_by_index(rl_idx, card)
                            #log(f"DEBUG — RL_Agent after apply: busted={player.busted}, stopped={player.stopped}, frozen={player.frozen}")

                            if player.busted:
                                player.stopped = True
                                leading_idx = engine.get_leading_opponent(rl_idx)
                                if leading_idx is not None:
                                    score_gap = engine.players[leading_idx].total_points - player.total_points
                                else:
                                    score_gap = 0

                                reward = compute_reward(
                                    points_banked=0,
                                    busted=True,
                                    score_gap=score_gap
                                )
                                update_q_table(state, action, reward, 
                                             next_state=state,   # terminal — ignored
                                             terminal=True)
                                stats["busts"] += 1
                            else:
                                # Survived the draw — no reward yet, just transition
                                next_state = get_state(player, engine, rl_idx)
                                update_q_table(state, action, reward=0.0,
                                             next_state=next_state,
                                             terminal=False)

                    else:
                        # --- Heuristic opponent turns --- # updated to call them by name instead of index for clarity during testing -4/1/26
                        opponent_name = engine.players[idx].name
                        if opponent_name == "Conservative":
                            action = conservative_policy(player)
                        elif opponent_name == "Greedy":
                            action = greedy_policy(player)
                        elif opponent_name == "Balanced":
                            action = balanced_policy(player)

                        if action == "STOP":
                            engine.bank_points(player)
                        elif action == "DRAW":
                            card = engine.draw_card()
                            engine.apply_card_by_index(idx, card)
                            if player.busted:
                                player.stopped = True

                if all_done:
                    round_over = True

            # --- End of round ---
            # Bank frozen players
            for player in engine.players:
                if player.frozen and not player.busted:
                    player.total_points += player.round_points * player.multiplier

            # Check win condition
            # DEBUG - Show totals after every round, missing totals contriubuting to .007 winrate - 4/2/26
            for player in engine.players:
                #log(f"DEBUG END ROUND — {player.name}: round={player.round_points}, total={player.total_points},stopped={player.stopped}, busted={player.busted}")
                if player.total_points >= win_threshold:
                    if player.name == "RL_Agent":
                        stats["wins"] += 1
                        # updated game_reward to 3.0 to give more weight to winning the game, previously 1.0 -4/3/26
                        game_reward += 3.0    # win bonus
                        # Apply win bonus update to last state
                        update_q_table(state, action, reward=1.0,
                                      next_state=state, terminal=True)
                    else:
                        stats["losses"] += 1
                    game_over = True
                    break

        # --- End of game ---
        recent_rewards.append(game_reward)
        if len(recent_rewards) > verbose_every:
            recent_rewards.pop(0)

        # Decay epsilon
        EPSILON = max(EPSILON_MIN, EPSILON * EPSILON_DECAY)

        # Snapshot logging
        if game_num % verbose_every == 0:
            win_rate = stats["wins"] / game_num
            avg_reward = sum(recent_rewards) / len(recent_rewards)
            stats["win_rate_log"].append(win_rate)
            stats["epsilon_log"].append(EPSILON)
            stats["avg_reward_log"].append(avg_reward)
            log(f"Game {game_num:6d} | " # Replaced with log instead of print for training log file -4/1/26
                  f"Win Rate: {win_rate:.3f} | "
                  f"Epsilon: {EPSILON:.4f} | "
                  f"Avg Reward: {avg_reward:.3f} | "
                  f"Busts: {stats['busts']}")

    return stats

In [ ]:
# let's actually run it
# Reset epsilon before training
EPSILON = 1.0

# Run training
training_stats = train_rl_agent(
    num_games=10000,
    num_opponents=3,
    win_threshold=200,
    verbose_every=1000
)

In [ ]:
import os
print(os.path.abspath('flip7_training_log.txt'))

In [ ]:
# Sample Q-table entries — what has the agent actually learned?
print(f"Total states learned: {len(q_table)}")
print("\nSample Q-values:")
for i, (state, values) in enumerate(list(q_table.items())[:20]):
    print(f"State {state}: DRAW={values['DRAW']:.4f}, STOP={values['STOP']:.4f}")

My Positive rewards are too small, vs negative. So it's choosing to stop rather than draw.

### Initial Thoughts after initial Testing
- The training results of 0.6% of wins in 10,000 games tells me there are issues with my rewards and Q-learning table
- Considering it was using a state for raw score - the explosion of states the agent could encounter would be around 6400 at a minimum, and we're only running 10,000 games for training, this means I need to adjust my Q-Table to give the agent data that it can utilize, so we'll switch to score bins.
- Essentially more dimensions with less visitation = worse learning.
- My initial thought is 4 bins would give me good enough details, but I don't think it's nuanced enough to see the difference between the stop points for the Conservative and Balanced player amounts.
- Additional issue, I'm not sure if the policies created for my other agents are configured appropriately. Will need to test just the 3 heuristics policies against each other.



In [ ]:
# Creating baseline test to evaluate the heuristic policies against each other without the RL agent, to confirm they are working as expected and to provide a benchmark for the RL agent's performance.

def run_heuristic_baseline(num_games: int = 5000) -> dict:
    """
    Runs heuristic agents against each other to establish baseline win rates.
    No RL agent involved.
    """
    results = {"Conservative": 0, "Greedy": 0, "Balanced": 0, "draws": 0}
    
    for _ in range(num_games):
        result = run_game(
            player_names=["Conservative", "Greedy", "Balanced"],
            policies={
                0: conservative_agent,
                1: greedy_agent,
                2: balanced_agent
            },
            win_threshold=200,
            verbose=False
        )
        winner = result["winner"]
        if winner in results:
            results[winner] += 1
        else:
            results["draws"] += 1
    
    print(f"\nBaseline Results over {num_games} games:")
    print(f"Conservative: {results['Conservative']/num_games:.3f} win rate")
    print(f"Greedy:       {results['Greedy']/num_games:.3f} win rate")
    print(f"Balanced:     {results['Balanced']/num_games:.3f} win rate")
    
    return results

In [ ]:
# run the agent testing
agent_testing_stats = run_heuristic_baseline()

Close - but Balanced only has a 26.6% win rate. I'd like to move this heuristic closer to 33%.
Let's run a detailed baseline to make a better decision on how to adjust the Balanced Heuristic.

In [ ]:
# detailed Baseline setup
def run_detailed_baseline(num_games: int = 5000) -> None:
    """
    Tracks behavioral metrics per heuristic to diagnose performance.
    """
    metrics = {
        "Conservative": {"wins": 0, "busts": 0, "total_banked": 0, "stops": 0, "rounds_played": 0},
        "Greedy":       {"wins": 0, "busts": 0, "total_banked": 0, "stops": 0, "rounds_played": 0},
        "Balanced":     {"wins": 0, "busts": 0, "total_banked": 0, "stops": 0, "rounds_played": 0}
    }

    for _ in range(num_games):
        engine = Flip7RoundEngine(["Conservative", "Greedy", "Balanced"])
        game_over = False

        while not game_over:
            engine.reset_round()

            while not engine.all_players_done():
                for idx, player in enumerate(engine.players):
                    if not player.is_active():
                        continue

                    name = player.name
                    metrics[name]["rounds_played"] += 1

                    if name == "Conservative":
                        action = conservative_policy(player)
                    elif name == "Greedy":
                        action = greedy_policy(player)
                    elif name == "Balanced":
                        action = balanced_policy(player)

                    if action == "STOP":
                        metrics[name]["stops"] += 1
                        metrics[name]["total_banked"] += player.round_points
                        engine.bank_points(player)
                    elif action == "DRAW":
                        card = engine.draw_card()
                        engine.apply_card_by_index(idx, card)
                        if player.busted:
                            metrics[name]["busts"] += 1
                            player.stopped = True

            for player in engine.players:
                if player.frozen and not player.busted:
                    player.total_points += player.round_points * player.multiplier

            for player in engine.players:
                if player.total_points >= 200:
                    if player.name in metrics:
                        metrics[player.name]["wins"] += 1
                    game_over = True
                    break

    print(f"\nDetailed Baseline Results over {num_games} games:")
    print(f"{'Metric':<25} {'Conservative':>12} {'Greedy':>12} {'Balanced':>12}")
    print("-" * 63)
    for metric in ["wins", "busts", "stops", "total_banked"]:
        vals = {k: metrics[k][metric] for k in metrics}
        print(f"{metric:<25} {vals['Conservative']:>12} {vals['Greedy']:>12} {vals['Balanced']:>12}")

    print("\nDerived Metrics:")
    print(f"{'Metric':<25} {'Conservative':>12} {'Greedy':>12} {'Balanced':>12}")
    print("-" * 63)

    for name in ["Conservative", "Greedy", "Balanced"]:
        m = metrics[name]
        avg_banked = m["total_banked"] / max(m["stops"], 1)
        bust_rate  = m["busts"] / max(m["rounds_played"], 1)
        stop_rate  = m["stops"] / max(m["rounds_played"], 1)
        print(f"Avg points banked per stop: {avg_banked:>6.1f}  "
              f"Bust rate: {bust_rate:>5.3f}  "
              f"Stop rate: {stop_rate:>5.3f}  "
              f"— {name}")

In [ ]:
run_detailed_baseline()

The rates don't look perfect, as Greedy is still winning this race - but, Balanced has moved closer to 30% and what we're really seeing in my opinion is Greedy benefitting from the dynamics of the game, which reward aggressive plays via special cards and high value cards.
I think the heuristic agents are as well tuned as I will get them for the time of this project, unless I need to adjust new policies.